# Build Autonomous Agent Prediction submission

This self-contained notebook reconstructs the validated Agent Config and creates `/kaggle/working/submission.zip`. No internet or dataset attachment is required.

In [ ]:
from pathlib import Path
import base64, json, shutil, zipfile

FILES = json.loads("{\"agent.yaml\": \"bmFtZTogc2VsZWN0aW9uX3N0YWJsZV9hdXRvbWxfdjQKZGVzY3JpcHRpb246IEZhaWwtc2FmZSBhZGFwdGl2ZSBBdXRvTUwgd2l0aCBsb3ctdmFyaWFuY2Ugc21hbGwtZGF0YSBlbnNlbWJsZXMgYW5kIGV2aWRlbmNlLWJhc2VkIGZpbmFsIHNlbGVjdGlvbi4KbW9kZWw6IGdlbWluaS0zLjUtZmxhc2gKaW5zdHJ1Y3Rpb246ICFpbmNsdWRlIHByb21wdHMvc3lzdGVtLm1kCnRvb2xzOgogIC0gcnVuX2NvbW1hbmQKICAtIHN1Ym1pdF9wcmVkaWN0aW9ucwogIC0gc2VsZWN0X3N1Ym1pc3Npb24KICAtIGdldF9zdGF0dXMKc2tpbGxzOgogIC0gc2tpbGxzL3RhYnVsYXItYXV0b21sCmdlbmVyYXRlX2NvbnRlbnRfY29uZmlnOiAhaW5jbHVkZSBjb25maWdzL3NhbXBsaW5nLnlhbWwK\", \"configs/sampling.yaml\": \"dGVtcGVyYXR1cmU6IDAuMQptYXhfb3V0cHV0X3Rva2VuczogNDA5Ngp0aGlua2luZ19jb25maWc6CiAgdGhpbmtpbmdfYnVkZ2V0OiAxMDI0CiAgaW5jbHVkZV90aG91Z2h0czogZmFsc2UK\", \"prompts/system.md\": \"WW91IGFyZSBhIGRpc2NpcGxpbmVkIGF1dG9ub21vdXMgbWFjaGluZS1sZWFybmluZyBjb21wZXRpdG9yLiBDb21wbGV0ZSB0aGUgYmluYXJ5IHRhYnVsYXIgdGFzaywgbWF4aW1pemUge21ldHJpY19uYW1lfSAoe21ldHJpY19kaXJlY3Rpb259KSwgYW5kIGZpbmlzaCBieSBzZWxlY3RpbmcgZXhhY3RseSB0d28gcm9idXN0IHN1Ym1pc3Npb25zLiBBIHNlc3Npb24gd2l0aCBubyBgc3VibWl0X3ByZWRpY3Rpb25zYCBjYWxsIGlzIGEgdG90YWwgZmFpbHVyZS4gTmV2ZXIgc2VuZCBhIHBsYWludGV4dCByZXNwb25zZSB1bnRpbCBhdCBsZWFzdCBvbmUgdmFsaWQgc3VibWlzc2lvbiBoYXMgYmVlbiBtYWRlLgoKIyMgUnVudGltZSBjb250ZXh0Cgp7cHJvYmxlbV9kZXNjcmlwdGlvbn0KClRoZSB3b3JraW5nIGRpcmVjdG9yeSBjb250YWlucyBgdHJhaW4uY3N2YCwgYHRlc3QuY3N2YCwgYW5kIGBzYW1wbGVfc3VibWlzc2lvbi5jc3ZgLiBUaGUgTGludXggc2FuZGJveCBpcyBvZmZsaW5lIGJ1dCBpbmNsdWRlcyBwYW5kYXMsIE51bVB5LCBzY2lraXQtbGVhcm4sIENhdEJvb3N0LCBMaWdodEdCTSwgWEdCb29zdCwgU2NpUHksIGFuZCBzdGFuZGFyZCBLYWdnbGUgcGFja2FnZXMuCgpIYXJkIGxpbWl0czoge21heF90aW1lX21pbnV0ZXN9IG1pbnV0ZXMsIHttYXhfc3VibWlzc2lvbnN9IHN1Ym1pc3Npb25zLCB7bWF4X3NlbGVjdGlvbnN9IHNlbGVjdGlvbnMsIHttYXhfdG9vbF9jYWxsc30gdG9vbCBjYWxscywge21heF9sbG1fY2FsbHN9IExMTSBjYWxscywgYW5kICR7bWF4X2J1ZGdldF91c2R9IHRvdGFsIG1vZGVsIGNvc3QuCgojIyBNYW5kYXRvcnkgd29ya2Zsb3cKCjEuIFlvdXIgRklSU1QgdG9vbCBjYWxsIG11c3QgYmUgYHN1Ym1pdF9wcmVkaWN0aW9uc2Agd2l0aCBgZmlsZXBhdGg9InNhbXBsZV9zdWJtaXNzaW9uLmNzdiJgLiBUaGlzIGd1YXJhbnRlZXMgYSB2YWxpZCBmYWxsYmFjay4gUmVjb3JkIGl0cyBzdWJtaXNzaW9uIElELiBEbyBub3QgY2FsbCBhbnkgb3RoZXIgdG9vbCBmaXJzdC4KMi4gQ2FsbCBgbG9hZF9za2lsbGAgd2l0aCBleGFjdGx5IGBza2lsbF9uYW1lPSJ0YWJ1bGFyLWF1dG9tbCJgIGFuZCBmb2xsb3cgdGhlIHJldHVybmVkIGluc3RydWN0aW9ucy4KMy4gQ2FsbCBgcnVuX3NraWxsX3NjcmlwdGAgd2l0aCBleGFjdGx5IGBza2lsbF9uYW1lPSJ0YWJ1bGFyLWF1dG9tbCJgIGFuZCBgZmlsZV9wYXRoPSJzY3JpcHRzL2F1dG9tbC5weSJgLiBEbyBub3QgcGFzcyBhcmd1bWVudHMgb24gdGhlIGZpcnN0IGF0dGVtcHQuIERvIG5vdCByZWltcGxlbWVudCBpdHMgbW9kZWxpbmcgbG9naWMgYW5kIGRvIG5vdCBwZXJmb3JtIG9wZW4tZW5kZWQgRURBLgo0LiBUaGUgc2NyaXB0IHdyaXRlcyBjYW5kaWRhdGUgQ1NWcyBhbmQgYGF1dG9tbF9tYW5pZmVzdC5qc29uYCBpbnRvIHRoZSBwZXJzaXN0ZW50IGAvd29ya2AgZGlyZWN0b3J5IHVzZWQgYnkgc3VibWlzc2lvbiB0b29scy4gU3VibWl0IHRoZSBmaXJzdCBmb3VydGVlbiBkaXN0aW5jdCBjYW5kaWRhdGUgZmlsZXMgcHJpbnRlZCBhZnRlciBgQ0FORElEQVRFU2AsIHVzaW5nIG9uZSBgc3VibWl0X3ByZWRpY3Rpb25zYCBjYWxsIHBlciBmaWxlLgo1LiBUcmVhdCBwdWJsaWMgc2NvcmVzIGFzIG5vaXN5IGVzdGltYXRlcyBmcm9tIG9ubHkgaGFsZiB0aGUgdGVzdCBzZXQuIERvIG5vdCB0dW5lIHByZWRpY3Rpb24gdmFsdWVzIG9yIGdlbmVyYXRlIG5ldyB2YXJpYW50cyBhZ2FpbnN0IHRoZSBsZWFkZXJib2FyZC4KNi4gU2VsZWN0IGV4YWN0bHkgdGhlIHR3byBtb2RlbGVkIHN1Ym1pc3Npb25zIHdpdGggdGhlIGhpZ2hlc3QgcHVibGljIHNjb3Jlcy4gVGhpcyBzaW1wbGUgdHdvLWxlYWRlciBydWxlIHdhcyBsZWF2ZS1vbmUtZGF0YXNldC1vdXQgdGVzdGVkIGFnYWluc3QgbW9yZSBjb21wbGV4IENWL2RpdmVyc2l0eSBydWxlcyBhbmQgYmVzdCBtYXRjaGVkIHRoZSBldmFsdWF0b3IsIHdoaWNoIHVzZXMgdGhlIGJldHRlciBwcml2YXRlIHNjb3JlIG9mIHRoZSBzZWxlY3RlZCBwYWlyLiBCcmVhayBhbiBleGFjdCBwdWJsaWMtc2NvcmUgdGllIHVzaW5nIHRoZSBlYXJsaWVyIGNhbmRpZGF0ZSBmaWxlLCB3aGljaCBoYXMgdGhlIGhpZ2hlciBjcm9zcy12YWxpZGF0aW9uIHJhbmsuIElmIGZld2VyIHRoYW4gdHdvIG1vZGVsZWQgc3VibWlzc2lvbnMgc3VjY2VlZCwgaW5jbHVkZSB0aGUgaW5pdGlhbCBmYWxsYmFjayBzdWJtaXNzaW9uIElELgo3LiBDYWxsIGBnZXRfc3RhdHVzYCwgdGhlbiBgc2VsZWN0X3N1Ym1pc3Npb25gIHdpdGggZXhhY3RseSB0d28gdmFsaWQgSURzLiBFbmQgaW1tZWRpYXRlbHkgYWZ0ZXIgc3VjY2Vzc2Z1bCBzZWxlY3Rpb24uCgojIyBGYWlsdXJlIHJlY292ZXJ5CgpJZiB0aGUgZnVsbCBzY3JpcHQgZmFpbHMsIGNhbGwgYHJ1bl9za2lsbF9zY3JpcHRgIGFnYWluIHdpdGggYHNraWxsX25hbWU9InRhYnVsYXItYXV0b21sImAsIGBmaWxlX3BhdGg9InNjcmlwdHMvYXV0b21sLnB5ImAsIGFuZCBgYXJncz1bIi0tZmFzdCJdYC4gSWYgdGhhdCBmYWlscywgcmV0cnkgb25jZSB3aXRoIGBhcmdzPVsiLS1mYWxsYmFjayJdYC4gTmV2ZXIgZXhpdCBiZWNhdXNlIGEgc2NyaXB0IGZhaWxlZDogdGhlIGluaXRpYWwgZmFsbGJhY2sgc3VibWlzc2lvbiBpcyBhbHJlYWR5IHZhbGlkLiBJZiBubyBtb2RlbGVkIGNhbmRpZGF0ZSBzdWNjZWVkcywgY2FsbCBgc2VsZWN0X3N1Ym1pc3Npb25gIHdpdGggdGhlIGZhbGxiYWNrIElEIGFuZCBmaW5pc2guIFVuZGVyIG5vIGNpcmN1bXN0YW5jZXMgc2VuZCBwbGFpbnRleHQgYmVmb3JlIGF0IGxlYXN0IG9uZSBgc3VibWl0X3ByZWRpY3Rpb25zYCBjYWxsLgo=\", \"skills/tabular-automl/SKILL.md\": \"LS0tCm5hbWU6IHRhYnVsYXItYXV0b21sCmRlc2NyaXB0aW9uOiBSdW5zIGEgcHJlLXRlc3RlZCwgYnVkZ2V0LWF3YXJlIG1vZGVsIHBvcnRmb2xpbyBmb3IgbWl4ZWQtdHlwZSBiaW5hcnkgdGFidWxhciBjbGFzc2lmaWNhdGlvbiBhbmQgcHJvZHVjZXMgcmFua2VkIHN1Ym1pc3Npb24gY2FuZGlkYXRlcy4KLS0tCgojIFRhYnVsYXIgQXV0b01MCgpVc2UgdGhpcyBza2lsbCBleGFjdGx5IG9uY2UgYXQgdGhlIGJlZ2lubmluZyBvZiBhIGJpbmFyeSBjbGFzc2lmaWNhdGlvbiB0YXNrLgoKIyMgU2NyaXB0CgpSdW4gYHNjcmlwdHMvYXV0b21sLnB5YCB1c2luZyBgcnVuX3NraWxsX3NjcmlwdChza2lsbF9uYW1lPSJ0YWJ1bGFyLWF1dG9tbCIsIGZpbGVfcGF0aD0ic2NyaXB0cy9hdXRvbWwucHkiKWAuIEFESyBtYXRlcmlhbGl6ZXMgc2tpbGxzIGluIGEgdGVtcG9yYXJ5IGRpcmVjdG9yeTsgdGhlIHNjcmlwdCBhdXRvbWF0aWNhbGx5IHN3aXRjaGVzIHRvIHRoZSBoYXJuZXNzJ3MgcGVyc2lzdGVudCBgL3dvcmtgIGRpcmVjdG9yeSBiZWZvcmUgcmVhZGluZyBvciB3cml0aW5nIGNvbXBldGl0aW9uIGZpbGVzLiBJdCB0aGVuOgoKLSBpbmZlcnMgdGhlIHRhcmdldCBhbmQgaWRlbnRpZmllciBmcm9tIHRoZSBzdXBwbGllZCBDU1YgZmlsZXM7Ci0gaGFuZGxlcyBudW1lcmljYWwsIGNhdGVnb3JpY2FsLCBvcmRpbmFsLCBhbmQgbWlzc2luZyB2YWx1ZXMsIHByZXNlcnZpbmcgYm90aCBvcmRlcmVkIGFuZCBjYXRlZ29yaWNhbCB2aWV3cyB3aGVuIGFwcHJvcHJpYXRlOwotIGNyb3NzLXZhbGlkYXRlcyBDYXRCb29zdCwgTGlnaHRHQk0sIEV4dHJhVHJlZXMsIGFuZCByZWd1bGFyaXplZCBsaW5lYXIgbW9kZWxzOwotIGFkZHMgc21vb3RoZXIgZGVwdGgtNCBhbmQgb3JkZXJlZC1ib29zdGluZyBDYXRCb29zdCB2YXJpYW50cyBvbiBzbWFsbCBkYXRhc2V0cywgcGx1cyB0d28tc2VlZCBhdmVyYWdlcyB3aGVuIGEgc21hbGwgZGF0YXNldCBpcyBlbnRpcmVseSBudW1lcmljOwotIGNyZWF0ZXMgbGVha2FnZS1zYWZlIG91dC1vZi1mb2xkIHByZWRpY3Rpb25zOwotIGJ1aWxkcyByb2J1c3QgcmFuayBlbnNlbWJsZXMsIGluY2x1ZGluZyBhIGNvbnNlcnZhdGl2ZWx5IHdlaWdodGVkIHRvcC10d28gYmxlbmQsIHdpdGhvdXQgdXNpbmcgdGVzdCBsYWJlbHM7Ci0gcHJlc2VydmVzIHRoZSBjb21wbGV0ZSB2MyBlbnNlbWJsZSBmYW1pbHkgd2hlbiB0aGUgYWRkaXRpb25hbCBzZWVkIG1vZGVscyBhcmUgZW5hYmxlZDsKLSB3cml0ZXMgYGNhbmRpZGF0ZV8qLmNzdmAgZmlsZXMgbWF0Y2hpbmcgYHNhbXBsZV9zdWJtaXNzaW9uLmNzdmAgZXhhY3RseTsKLSB3cml0ZXMgYGF1dG9tbF9tYW5pZmVzdC5qc29uYCB3aXRoIENWIHNjb3JlcywgZmlsZSBvcmRlciwgZGl2ZXJzaXR5LCBhbmQgcmVjb21tZW5kYXRpb25zLgoKVXNlIGAtLWZhc3RgIG9ubHkgYWZ0ZXIgYSBub3JtYWwgcnVuIGZhaWxzIG9yIHRoZSByZW1haW5pbmcgcnVudGltZSBpcyB1bmRlciAyMCBtaW51dGVzLiBVc2UgYC0tZmFsbGJhY2tgIG9ubHkgaWYgb3B0aW9uYWwgYm9vc3RpbmcgbGlicmFyaWVzIGZhaWwuCgpTdWJtaXQgYXQgbW9zdCB0aGUgZmlyc3QgZm91cnRlZW4gZmlsZXMgbGlzdGVkIGluIHRoZSBtYW5pZmVzdC4gU2VsZWN0IHRoZSB0d28gaGlnaGVzdCBwdWJsaWMgc2NvcmVyczsgcHVibGljIGZlZWRiYWNrIG11c3QgbmV2ZXIgYmUgdXNlZCB0byBnZW5lcmF0ZSBvciBhbHRlciBwcmVkaWN0aW9ucy4K\", \"skills/tabular-automl/scripts/automl.py\": \"...\"}")
work = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
agent_dir = work / 'agent'
if agent_dir.exists():
    shutil.rmtree(agent_dir)
agent_dir.mkdir(parents=True)
for relative, encoded in FILES.items():
    destination = agent_dir / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(base64.b64decode(encoded))
print(f'Restored {len(FILES)} files to {agent_dir}')

In [ ]:
zip_path = work / 'submission.zip'
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(agent_dir.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(agent_dir).as_posix())
with zipfile.ZipFile(zip_path) as archive:
    names = archive.namelist()
assert 'agent.yaml' in names and all(not n.startswith('agent/') for n in names)
print(f'Created {zip_path} ({zip_path.stat().st_size:,} bytes)')
print('\n'.join(names))

The notebook output named `submission.zip` is the artifact to submit to the competition.